# 03 — Marker scores, not cell proportions

Background-corrected marker expression provides exploratory cell-type-associated scores. Scores may be negative and do not sum to one. Reference Cell2location inference is deliberately unsupported until a validated reference workflow is available. Summaries are per biopsy; no spot-level disease tests are performed.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from src import preprocessing, utils, visualization
from src.spatial import build_spatial_graph, validate_graph, spatial_lr
settings = utils.load_settings()
params = settings['params']
seed = params['seed']
sc.settings.set_figure_params(dpi=params['visualization']['figure_dpi'], facecolor='white')

In [ ]:
run = utils.begin_stage(settings, 'adata_scored.h5ad', 'adata_clustered.h5ad')
adata = sc.read_h5ad(run / 'processed' / 'adata_clustered.h5ad')
utils.score_cell_types(adata, method=params['cell_types']['method'], seed=seed)
score_cols = list(adata.uns['marker_score_columns'])
summary = utils.sample_score_summary(adata, score_cols)
summary.to_csv(run / 'tables' / 'marker_scores_by_sample.csv', index=False)
adata.obs[['sample', 'condition', 'rejection_type'] + score_cols].to_csv(run / 'tables' / 'spot_marker_scores.csv')
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(summary.set_index('sample')[score_cols].T, center=0, cmap='RdBu_r', ax=ax)
ax.set_title('Mean marker score per biopsy — not cell fractions')
fig.tight_layout()
fig.savefig(run / 'figures' / 'marker_scores_by_sample.png')
plt.show()

In [ ]:
sample = str(adata.obs['sample'].iloc[0])
visualization.plot_spatial_multi_feature(adata, score_cols[:8], sample=sample, ncols=4, cmap='RdBu_r', save_path=run / 'figures' / 'marker_scores_on_tissue.png')
plt.show()
utils.save_checkpoint(adata, run / 'processed' / 'adata_scored.h5ad')